In [ ]:
from datetime import datetime
from getpass import getpass

rdm_url = 'https://rdm.example.com/'
idp_name_1 = None # 'GakuNin RDM IdP'
idp_username_1 = None
idp_user_display_name_1 = None
idp_password_1 = None
idp_name_2 = None # 'GakuNin RDM IdP'
idp_username_2 = None
idp_user_display_name_2 = None
idp_password_2 = None
project_name = None
delete_project = True
default_result_path = None
close_on_fail = False
transition_timeout = 30000

In [ ]:
import tempfile

if idp_username_1 is None:
    idp_username_1 = input(prompt=f'Username for Existing User 1 ({idp_name_1})')
if idp_user_display_name_1 is None:
    idp_user_display_name_1 = input(prompt=f'Display name for Existing User 1 ({idp_name_1})')
if idp_password_1 is None:
    idp_password_1 = getpass(prompt=f'Password for {idp_username_1}@{idp_name_1}')

if idp_username_2 is None:
    idp_username_2 = input(prompt=f'Username for Existing User 2 ({idp_name_2})')
if idp_user_display_name_2 is None:
    idp_user_display_name_2 = input(prompt=f'Display name for Existing User 2 ({idp_name_2})')
if idp_password_2 is None:
    idp_password_2 = getpass(prompt=f'Password for {idp_username_2}@{idp_name_2}')

if project_name is None:
    project_name = datetime.now().strftime('TEST-ScreenDisplay-%Y%m%d-Contributor')


# メンバー画面の表示項目の確認

- サブシステム名: 画面表示
- ページ/アドオン: Member
- 機能分類: 画面表示
- シナリオ名: メンバー画面の表示項目確認（プロジェクト作成者/招待されたメンバー双方の視点）
- 用意するテストデータ: URL一覧、アカウント(既存ユーザー1, 既存ユーザー2)
- 事前条件: 既存ユーザー1、既存ユーザー2のアカウントが利用可能であること

In [ ]:
work_dir = tempfile.mkdtemp()
if default_result_path is None:
    default_result_path = work_dir
work_dir

In [ ]:
import importlib
import pandas as pd

import scripts.playwright
importlib.reload(scripts.playwright)

from scripts.playwright import *
from scripts import grdm

await init_pw_context(close_on_fail=close_on_fail, last_path=default_result_path)

## ウェブブラウザの新規プライベートウィンドウでGRDMトップページを表示する

GRDMトップページが表示されること

In [ ]:
async def _step(page):
    await page.goto(rdm_url)

    # 同意する をクリック
    await page.locator('//button[text() = "同意する"]').click()

    # 同意する が表示されなくなったことを確認
    await expect(page.locator('//button[text() = "同意する"]')).to_have_count(0, timeout=500)

await run_pw(_step, new_context=True, new_page=True)

## 「GakuNinRDM IdP」を利用し、既存ユーザー1としてログインする

GRDMダッシュボードが表示されること

In [ ]:
async def _step(page):
    await grdm.login(page, idp_name_1, idp_username_1, idp_password_1, transition_timeout=transition_timeout)
    await grdm.expect_dashboard(page, transition_timeout=transition_timeout)

await run_pw(_step)

## プロジェクト一覧に指定されたタイトルのプロジェクトがない場合、指定された名前でプロジェクトを作成する。 ダッシュボードから「新規プロジェクト作成」をクリックする。 タイトル「TEST-ScreenDisplay-YYYYMMDD(本日の日付)-Contributor」でプロジェクトを作成する


- 「新規プロジェクトの作成」のダイアログが表示されること
- ダッシュボードのプロジェクト一覧に入力したプロジェクト名のプロジェクトが追加されること

In [ ]:
async def _step(page):
    await grdm.ensure_project_exists(page, project_name, transition_timeout=transition_timeout)

await run_pw(_step)

## ダッシュボードのプロジェクト一覧から作成したプロジェクトをクリックする

作成したプロジェクトのプロジェクトダッシュボードが表示されること

In [ ]:
project_url = None

async def _step(page):
    global project_url
    await page.locator(f'//*[@data-test-dashboard-item-title and text() = "{project_name}"]').click()
    await expect(page.locator('//span[@id = "nodeTitleEditable"]')).to_be_visible(timeout=transition_timeout)
    project_url = page.url

await run_pw(_step)

## プロジェクトダッシュボードの上部メニューから「メンバー」をクリックする

- 「メンバー」画面が表示されること
- 「メンバー」にプロジェクト作成者自身が「管理者」権限で表示されること
- 「メンバー」に「名前」「E-mail」「所属機関」「招待日」「権限」「目録表示」の項目が表示されること

In [ ]:
async def _step(page):
    await page.locator('#projectSubnav').get_by_role('link', name='メンバー').click()
    await expect(page.locator('#manageContributors')).to_be_visible(timeout=transition_timeout)

    creator_row = page.locator('#manageContributorsTable').locator(
        f'//tr[.//*[contains(@class, "name-search") and normalize-space(text())="{idp_user_display_name_1}"]]'
    )
    await expect(creator_row.locator('td.permissions select')).to_have_value('admin', timeout=transition_timeout)

    for header_text in ['名前', '権限', '目録表示', 'E-mail', '所属機関', '招待日']:
        await expect(page.locator('#manageContributorsTable').get_by_text(header_text, exact=False).first).to_be_visible(timeout=transition_timeout)

await run_pw(_step)

## 「メンバー」のタイトルの横にある「＋追加」をクリックする

「メンバーを追加」ダイアログが表示されること

In [ ]:
async def _step(page):
    await page.locator('a[href="#addContributors"]').click()
    await expect(page.locator('#addContributors')).to_be_visible(timeout=transition_timeout)
    await expect(page.locator('#addContributors .modal-title')).to_have_text('メンバーを追加', timeout=transition_timeout)

await run_pw(_step)

## 既存ユーザー2の情報(GUID,メールアドレス,名前)を入力して、「検索」ボタンをクリックする

「結果」の一覧に既存ユーザー2が表示されること

In [ ]:
async def _step(page):
    search_box = page.locator('#addContributors .input-group input.form-control')
    # テスト仕様書の「GUID, メールアドレス, 名前」のうち、名前(idp_user_display_name_2)で検索する
    await search_box.fill(idp_user_display_name_2)
    await page.locator('#addContributors input[type="submit"][value="検索"]').click()

    # 検索結果一覧は contributor.fullname で表示される（ログインユーザー名とは異なる場合がある）
    result_row = page.locator('#addContributors').locator(
        f'//div[contains(@class, "col-md-4")]//tr[.//span[text()="{idp_user_display_name_2}"]]'
    )
    await expect(result_row).to_be_visible(timeout=transition_timeout)

await run_pw(_step)

## 「結果」の一覧の既存ユーザー2の左の「＋」ボタンをクリックする

「追加中」の一覧に既存ユーザー2が表示されること

In [ ]:
async def _step(page):
    result_row = page.locator('#addContributors').locator(
        f'//div[contains(@class, "col-md-4")]//tr[.//span[text()="{idp_user_display_name_2}"]]'
    )
    await result_row.locator('a.contrib-button.btn-success').click()

    adding_row = page.locator('#addContributors').locator(
        f'//div[contains(@class, "col-md-8")]//tr[.//span[text()="{idp_user_display_name_2}"]]'
    )
    await expect(adding_row).to_be_visible(timeout=transition_timeout)

await run_pw(_step)

## 「権限」を「読込み/書込み」に設定して、「追加」ボタンをクリックする

「メンバー」画面に既存ユーザー2が追加されること

In [ ]:
async def _step(page):
    adding_row = page.locator('#addContributors').locator(
        f'//div[contains(@class, "col-md-8")]//tr[.//span[text()="{idp_user_display_name_2}"]]'
    )
    # メンバー追加ダイアログの permissionList は {value, text} のオブジェクト配列で optionsValue がないため、
    # <option> に value 属性が付かない。並び順 read / write / admin の index=1（読込み / 書込み）を選択する。
    await adding_row.locator('select').select_option(index=1)

    await page.locator('#addContributors .modal-footer').locator(
        '//a[contains(@class, "btn-success") and contains(text(), "追加")]'
    ).click()

    await expect(page.locator('#addContributors')).to_be_hidden(timeout=transition_timeout)

    # メンバー一覧側も contributor.shortname（≒フルネーム）で表示されるため、idp_user_display_name_2 で特定する。
    member_row = page.locator('#manageContributorsTable').locator(
        f'//tr[.//*[contains(@class, "name-search") and normalize-space(text())="{idp_user_display_name_2}"]]'
    )
    await expect(member_row).to_be_visible(timeout=transition_timeout)

    # 既存ユーザー2が「読込み/書込み」権限で追加されたこと
    await expect(member_row.locator('td.permissions select')).to_have_value('write', timeout=transition_timeout)

await run_pw(_step)

## ユーザーメニューから「ログアウト」を選択する

GRDMトップページが表示されること

In [ ]:
async def _step(page):
    await grdm.logout(page, idp_name_1, transition_timeout=transition_timeout)

await run_pw(_step)

## ウェブブラウザの新規プライベートウィンドウでGRDMトップページを表示する

GRDMトップページが表示されること

In [ ]:
async def _step(page):
    await page.goto(rdm_url)

    # 同意する をクリック
    await page.locator('//button[text() = "同意する"]').click()

    # 同意する が表示されなくなったことを確認
    await expect(page.locator('//button[text() = "同意する"]')).to_have_count(0, timeout=500)

await run_pw(_step, new_context=True, new_page=True)

## 「GakuNinRDM IdP」を利用し、既存ユーザー2としてログインする

GRDMダッシュボードが表示されること

In [ ]:
async def _step(page):
    await grdm.login(page, idp_name_2, idp_username_2, idp_password_2, transition_timeout=transition_timeout)
    await grdm.expect_dashboard(page, transition_timeout=transition_timeout)

await run_pw(_step)

## ダッシュボードのプロジェクト一覧からNo.3で作成したプロジェクトをクリックする

作成したプロジェクトのプロジェクトダッシュボードが表示されること

In [ ]:
async def _step(page):
    await page.locator(f'//*[@data-test-dashboard-item-title and text() = "{project_name}"]').click()
    await expect(page.locator('//span[@id = "nodeTitleEditable"]')).to_be_visible(timeout=transition_timeout)

await run_pw(_step)

## プロジェクトダッシュボードの上部メニューから「メンバー」をクリックする

- 「メンバー」画面が表示されること
- 「メンバー」に自身が「読込み/書込み」権限で表示されること
- 「メンバー」に「名前」「権限」「目録表示」の項目が表示されること

In [ ]:
async def _step(page):
    await page.locator('#projectSubnav').get_by_role('link', name='メンバー').click()
    await expect(page.locator('#manageContributors')).to_be_visible(timeout=transition_timeout)

    # 自身が「読込み/書込み」権限で表示されること
    # メンバー一覧は contributor.shortname（≒フルネーム）で表示されるため、idp_user_display_name_2 で特定する。
    member_row = page.locator('#manageContributorsTable').locator(
        f'//tr[.//*[contains(@class, "name-search") and normalize-space(text())="{idp_user_display_name_2}"]]'
    )
    await expect(member_row).to_be_visible(timeout=transition_timeout)

    await expect(
        member_row.locator('td.permissions').get_by_text('読込み / 書込み', exact=True)
    ).to_be_visible(timeout=transition_timeout)

    # 「名前」「権限」「目録表示」の項目が表示されること
    for header_text in ['名前', '権限', '目録表示']:
        await expect(page.locator('#manageContributorsTable').get_by_text(header_text, exact=False).first).to_be_visible(timeout=transition_timeout)
    # 読込み/書込み権限では「E-mail」「所属機関」「招待日」列は表示されない（contributors.mako で管理者のみ出力）
    for header_text in ['E-mail', '所属機関', '招待日']:
        await expect(
            page.locator('#manageContributorsTable thead th').filter(has_text=header_text)
        ).to_have_count(0, timeout=transition_timeout)

await run_pw(_step)

## ユーザーメニューから「ログアウト」を選択する

GRDMトップページが表示されること

In [ ]:
async def _step(page):
    await grdm.logout(page, idp_name_2, transition_timeout=transition_timeout)

await run_pw(_step)

## (指定がある場合) 既存ユーザー1としてログインし、プロジェクトを削除する。 ウェブブラウザの新規プライベートウィンドウでGRDMトップページを表示し、既存ユーザー1としてログインする。 作成したプロジェクトを開き、「設定」から「プロジェクトを削除」を実行する

- プロジェクトが削除されること

In [ ]:
async def _step(page):
    if not delete_project:
        return
    await page.goto(rdm_url)

    # 同意する をクリック
    await page.locator('//button[text() = "同意する"]').click()
    await expect(page.locator('//button[text() = "同意する"]')).to_have_count(0, timeout=500)

    # プロジェクトの削除は管理者権限が必要なため、既存ユーザー1でログインする
    await grdm.login(page, idp_name_1, idp_username_1, idp_password_1, transition_timeout=transition_timeout)
    await grdm.expect_dashboard(page, transition_timeout=transition_timeout)

    await page.locator(f'//*[@data-test-dashboard-item-title and text() = "{project_name}"]').click()
    await expect(page.locator('//span[@id = "nodeTitleEditable"]')).to_be_visible(timeout=transition_timeout)

    await grdm.delete_project(page, transition_timeout=transition_timeout)
    await grdm.expect_dashboard(page, transition_timeout=transition_timeout)

    # プロジェクトが削除され、ダッシュボードのプロジェクト一覧に表示されないこと
    await expect(
        page.locator(f'//*[@data-test-dashboard-item-title and text() = "{project_name}"]')
    ).to_have_count(0, timeout=transition_timeout)

await run_pw(_step, new_context=delete_project, new_page=delete_project)

In [ ]:
await finish_pw_context(screenshot=False, last_path=default_result_path)

In [ ]:
!rm -fr {work_dir}